# Descriptive and Exploratory Analysis

## Breast Cancer Five-Year Recurrence Prediction Study

This notebook describes the final study cohort before model development. It presents the outcome distribution, participant characteristics, missing information, continuous and categorical predictor distributions, and exploratory comparisons between patients with and without five-year recurrence.

The analyses in this notebook are descriptive and will not be used to select predictors before model validation. Feature selection and data preprocessing will be performed within the model-validation pipeline.

In [ ]:
# ============================================================
# Import required libraries
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (
    chi2_contingency,
    fisher_exact,
    mannwhitneyu
)


# Configure pandas display.
pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    140
)

pd.set_option(
    "display.max_colwidth",
    None
)


# Configure figure appearance.
sns.set_theme(
    style="whitegrid",
    context="notebook"
)


RANDOM_SEED = 20260912


print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# ============================================================
# Define project paths
# ============================================================

from pathlib import Path


PROJECT_DIRECTORY = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

CODE_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Code"
)

RAW_DATA_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Data"
    / "Raw"
)

PROCESSED_DATA_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Data"
    / "Processed"
)

DOCUMENTATION_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Documentation"
)

MANUSCRIPT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Manuscript"
)

RESULTS_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Results"
)

REVIEWER_MATERIALS_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Reviewer_Materials"
)


TABLE_DIRECTORY = (
    RESULTS_DIRECTORY
    / "Tables"
)

FIGURE_DIRECTORY = (
    RESULTS_DIRECTORY
    / "Figures"
)


TABLE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_MODELING_FILE = (
    PROCESSED_DATA_DIRECTORY
    / "final_modeling_dataset.xlsx"
)


print("Project directory:")
print(PROJECT_DIRECTORY)

print("\nFinal modelling dataset:")
print(FINAL_MODELING_FILE)

print("\nDescriptive tables will be saved in:")
print(TABLE_DIRECTORY)

print("\nFigures will be saved in:")
print(FIGURE_DIRECTORY)

In [ ]:
# ============================================================
# Load and verify the final modelling dataset
# ============================================================

if not FINAL_MODELING_FILE.exists():
    raise FileNotFoundError(
        "The final modelling dataset was not found:\n"
        f"{FINAL_MODELING_FILE}"
    )


analysis_data = pd.read_excel(
    FINAL_MODELING_FILE
)


expected_columns = [
    "Age_at_Diagnosis",
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Tumor_Size_mm",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy",
    "Recurrence"
]


assert analysis_data.shape == (1370, 18), (
    "Expected 1,370 patients and 18 variables."
)

assert (
    analysis_data.columns.tolist()
    == expected_columns
), (
    "The variables or their order differ from the "
    "verified final dataset."
)

assert (
    analysis_data["Recurrence"]
    .value_counts()
    .to_dict()
    == {
        "No": 1297,
        "Yes": 73
    }
), "The recurrence distribution is incorrect."

assert (
    analysis_data["Tumor_Size_mm"]
    .isna()
    .sum()
    == 174
), "The tumor-size missing count is incorrect."

assert (
    analysis_data.duplicated().sum()
    == 0
), "Completely duplicated records were detected."


print("FINAL DATASET LOADED AND VERIFIED")
print("---------------------------------")
print("Patients:", analysis_data.shape[0])
print(
    "Candidate predictors:",
    analysis_data.shape[1] - 1
)
print("Outcome variable: Recurrence")
print(
    "Observed tumor-size measurements:",
    analysis_data["Tumor_Size_mm"].notna().sum()
)
print(
    "Missing tumor-size measurements:",
    analysis_data["Tumor_Size_mm"].isna().sum()
)


print("\nFirst five records:")

display(
    analysis_data.head()
)

In [ ]:
# ============================================================
# Cohort and five-year recurrence summary
# ============================================================

OUTCOME_COLUMN = "Recurrence"


total_patients = len(analysis_data)

recurrence_count = int(
    analysis_data[OUTCOME_COLUMN]
    .eq("Yes")
    .sum()
)

no_recurrence_count = int(
    analysis_data[OUTCOME_COLUMN]
    .eq("No")
    .sum()
)

missing_outcome_count = int(
    analysis_data[OUTCOME_COLUMN]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Calculate recurrence prevalence
# ------------------------------------------------------------

recurrence_proportion = (
    recurrence_count
    / total_patients
)

recurrence_percentage = (
    recurrence_proportion
    * 100
)


# ------------------------------------------------------------
# Calculate 95% Wilson confidence interval
# ------------------------------------------------------------

z_value = 1.96

wilson_denominator = (
    1
    + (z_value ** 2 / total_patients)
)

wilson_centre = (
    recurrence_proportion
    + (
        z_value ** 2
        / (2 * total_patients)
    )
)

wilson_adjustment = z_value * np.sqrt(
    (
        recurrence_proportion
        * (1 - recurrence_proportion)
        / total_patients
    )
    +
    (
        z_value ** 2
        / (4 * total_patients ** 2)
    )
)


recurrence_ci_lower = (
    wilson_centre
    - wilson_adjustment
) / wilson_denominator


recurrence_ci_upper = (
    wilson_centre
    + wilson_adjustment
) / wilson_denominator


recurrence_ci_lower_percentage = (
    recurrence_ci_lower
    * 100
)

recurrence_ci_upper_percentage = (
    recurrence_ci_upper
    * 100
)


# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

cohort_summary = pd.DataFrame({
    "Measure": [
        "Total patients",
        "No recurrence",
        "Recurrence",
        "Missing recurrence outcome",
        "Recurrence prevalence",
        "95% confidence interval"
    ],
    "Result": [
        f"{total_patients:,}",
        (
            f"{no_recurrence_count:,} "
            f"({no_recurrence_count / total_patients * 100:.2f}%)"
        ),
        (
            f"{recurrence_count:,} "
            f"({recurrence_percentage:.2f}%)"
        ),
        f"{missing_outcome_count:,}",
        f"{recurrence_percentage:.2f}%",
        (
            f"{recurrence_ci_lower_percentage:.2f}%"
            f"–{recurrence_ci_upper_percentage:.2f}%"
        )
    ]
})


# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert total_patients == 1370

assert recurrence_count == 73

assert no_recurrence_count == 1297

assert missing_outcome_count == 0

assert (
    recurrence_count
    + no_recurrence_count
    == total_patients
)


print("COHORT AND OUTCOME SUMMARY")
print("--------------------------")

display(cohort_summary)


print(
    "recurrence prevalence:",
    (
        f"{recurrence_percentage:.2f}% "
        f"(95% CI: "
        f"{recurrence_ci_lower_percentage:.2f}%–"
        f"{recurrence_ci_upper_percentage:.2f}%)"
    )
)

In [ ]:
# ============================================================
# Table 1: Continuous-variable section
# ============================================================

OUTCOME_COLUMN = "Recurrence"


no_recurrence_data = analysis_data.loc[
    analysis_data[OUTCOME_COLUMN].eq("No")
].copy()

recurrence_data = analysis_data.loc[
    analysis_data[OUTCOME_COLUMN].eq("Yes")
].copy()


def format_median_iqr(values):
    """
    Format a numeric variable as median [Q1-Q3].
    """

    numeric_values = pd.to_numeric(
        values,
        errors="coerce"
    ).dropna()

    median = numeric_values.median()
    first_quartile = numeric_values.quantile(0.25)
    third_quartile = numeric_values.quantile(0.75)

    return (
        f"{median:.1f} "
        f"[{first_quartile:.1f}-{third_quartile:.1f}]"
    )


def format_p_value(p_value):
    """
    Format p-values for publication.
    """

    if p_value < 0.001:
        return "<0.001"

    return f"{p_value:.3f}"


continuous_variable_labels = {
    "Age_at_Diagnosis": "Age at diagnosis, years",
    "Tumor_Size_mm": "Tumour size, mm"
}


continuous_table_rows = []
continuous_test_records = []


for variable, label in continuous_variable_labels.items():

    overall_values = pd.to_numeric(
        analysis_data[variable],
        errors="coerce"
    ).dropna()

    no_recurrence_values = pd.to_numeric(
        no_recurrence_data[variable],
        errors="coerce"
    ).dropna()

    recurrence_values = pd.to_numeric(
        recurrence_data[variable],
        errors="coerce"
    ).dropna()

    test_result = mannwhitneyu(
        no_recurrence_values,
        recurrence_values,
        alternative="two-sided"
    )

    continuous_table_rows.append({
        "Characteristic": (
            f"{label}, median [IQR]"
        ),
        "No recurrence (N=1,297)": (
            format_median_iqr(
                no_recurrence_values
            )
        ),
        "Recurrence (N=73)": (
            format_median_iqr(
                recurrence_values
            )
        ),
        "Overall (N=1,370)": (
            format_median_iqr(
                overall_values
            )
        ),
        "P value": format_p_value(
            test_result.pvalue
        )
    })

    continuous_test_records.append({
        "Variable": label,
        "Overall observed": len(overall_values),
        "No recurrence observed": len(
            no_recurrence_values
        ),
        "Recurrence observed": len(
            recurrence_values
        ),
        "Missing": int(
            analysis_data[variable]
            .isna()
            .sum()
        ),
        "Test": "Mann-Whitney U",
        "Raw p value": test_result.pvalue
    })


continuous_table1 = pd.DataFrame(
    continuous_table_rows
)

continuous_test_summary = pd.DataFrame(
    continuous_test_records
)


print("TABLE 1: CONTINUOUS VARIABLES")

display(continuous_table1)


print("\nCONTINUOUS-VARIABLE TEST DETAILS")

display(continuous_test_summary)

In [ ]:
# ============================================================
# Table 1: Categorical-variable section
# ============================================================

categorical_variable_labels = {
    "Marital_Status": "Marital status",
    "Nationality": "Nationality",
    "Menopause_Status": "Menopause status",
    "Breast_Cancer_Family_History": (
        "Breast cancer family history"
    ),
    "Primary_Site_Group": "Primary tumour site",
    "Laterality": "Laterality",
    "Histology_Group": "Histology",
    "Tumor_Grade": "Tumour grade",
    "Summary_Stage": "Summary stage",
    "ER_Status": "Estrogen receptor status",
    "PR_Status": "Progesterone receptor status",
    "HER2_Status": "HER2 status",
    "Surgery": "Surgery",
    "Chemotherapy": "Chemotherapy",
    "Hormone_Therapy": "Hormone therapy"
}


category_orders = {
    "Marital_Status": [
        "Married",
        "Single",
        "Widowed",
        "Divorced",
        "Unknown"
    ],
    "Nationality": [
        "Saudi",
        "Non_Saudi"
    ],
    "Menopause_Status": [
        "Pre",
        "Post",
        "Not_Applicable",
        "Unknown"
    ],
    "Breast_Cancer_Family_History": [
        "No",
        "Yes",
        "Unknown"
    ],
    "Primary_Site_Group": [
        "Breast_NOS",
        "Central_or_Nipple",
        "Inner_Quadrant",
        "Outer_Quadrant",
        "Overlapping_Lesion"
    ],
    "Laterality": [
        "Left",
        "Right",
        "Unknown"
    ],
    "Histology_Group": [
        "In_Situ",
        "Invasive_Ductal_NST",
        "Invasive_Lobular",
        "Mixed_Invasive",
        "Other_Invasive"
    ],
    "Tumor_Grade": [
        "Grade_1",
        "Grade_2",
        "Grade_3",
        "Unknown"
    ],
    "Summary_Stage": [
        "In_Situ",
        "Localized",
        "Regional",
        "Distant",
        "Unknown"
    ],
    "ER_Status": [
        "Negative",
        "Positive",
        "Unknown"
    ],
    "PR_Status": [
        "Negative",
        "Positive",
        "Unknown"
    ],
    "HER2_Status": [
        "Negative_or_Equivocal",
        "Positive",
        "Unknown"
    ],
    "Surgery": [
        "No",
        "Yes"
    ],
    "Chemotherapy": [
        "No",
        "Yes"
    ],
    "Hormone_Therapy": [
        "No",
        "Yes"
    ]
}


category_display_labels = {
    "Non_Saudi": "Non-Saudi",
    "Not_Applicable": "Not applicable",
    "Breast_NOS": "Breast, NOS",
    "Central_or_Nipple": "Central portion or nipple",
    "Inner_Quadrant": "Inner quadrant",
    "Outer_Quadrant": "Outer quadrant",
    "Overlapping_Lesion": "Overlapping lesion",
    "In_Situ": "In situ",
    "Invasive_Ductal_NST": "Invasive ductal/NST",
    "Invasive_Lobular": "Invasive lobular",
    "Mixed_Invasive": "Mixed invasive",
    "Other_Invasive": "Other invasive",
    "Grade_1": "Grade 1",
    "Grade_2": "Grade 2",
    "Grade_3": "Grade 3",
    "Negative_or_Equivocal": "Negative or equivocal"
}


def format_count_percentage(count, denominator):
    """
    Format a categorical result as n (%).
    """

    percentage = count / denominator * 100

    return f"{count:,} ({percentage:.1f}%)"


def permutation_chi_square_test(
    predictor,
    outcome,
    number_of_permutations=10000,
    random_seed=20260912
):
    """
    Calculate a permutation p-value for a sparse
    categorical contingency table larger than 2 x 2.
    """

    predictor_codes, levels = pd.factorize(
        predictor,
        sort=True
    )

    outcome_codes = (
        outcome.eq("Yes")
        .astype(int)
        .to_numpy()
    )

    number_of_levels = len(levels)

    observed_table = np.bincount(
        predictor_codes * 2 + outcome_codes,
        minlength=number_of_levels * 2
    ).reshape(number_of_levels, 2)

    row_totals = observed_table.sum(
        axis=1,
        keepdims=True
    )

    column_totals = observed_table.sum(
        axis=0,
        keepdims=True
    )

    total = observed_table.sum()

    expected_table = (
        row_totals
        @ column_totals
        / total
    )

    observed_statistic = np.sum(
        (
            observed_table
            - expected_table
        ) ** 2
        / expected_table
    )

    random_generator = np.random.default_rng(
        random_seed
    )

    exceedance_count = 0

    for _ in range(number_of_permutations):

        permuted_outcome = (
            random_generator.permutation(
                outcome_codes
            )
        )

        permuted_table = np.bincount(
            predictor_codes * 2
            + permuted_outcome,
            minlength=number_of_levels * 2
        ).reshape(number_of_levels, 2)

        permuted_statistic = np.sum(
            (
                permuted_table
                - expected_table
            ) ** 2
            / expected_table
        )

        if permuted_statistic >= observed_statistic:
            exceedance_count += 1

    permutation_p_value = (
        exceedance_count + 1
    ) / (
        number_of_permutations + 1
    )

    return permutation_p_value


def select_categorical_test(data, predictor):
    """
    Select chi-square, Fisher's exact, or a permutation
    chi-square test based on table size and expected counts.
    """

    contingency_table = pd.crosstab(
        data[predictor],
        data[OUTCOME_COLUMN]
    )

    chi_square_result = chi2_contingency(
        contingency_table,
        correction=False
    )

    expected_counts = chi_square_result.expected_freq

    minimum_expected = expected_counts.min()

    percentage_expected_at_least_five = (
        (expected_counts >= 5).mean()
        * 100
    )

    assumptions_met = (
        minimum_expected >= 1
        and percentage_expected_at_least_five >= 80
    )

    if assumptions_met:

        return {
            "p_value": chi_square_result.pvalue,
            "test": "Pearson chi-square",
            "minimum_expected": minimum_expected,
            "percentage_at_least_five": (
                percentage_expected_at_least_five
            )
        }

    if contingency_table.shape == (2, 2):

        fisher_result = fisher_exact(
            contingency_table.to_numpy()
        )

        return {
            "p_value": fisher_result.pvalue,
            "test": "Fisher's exact",
            "minimum_expected": minimum_expected,
            "percentage_at_least_five": (
                percentage_expected_at_least_five
            )
        }

    permutation_p_value = (
        permutation_chi_square_test(
            predictor=data[predictor],
            outcome=data[OUTCOME_COLUMN]
        )
    )

    return {
        "p_value": permutation_p_value,
        "test": (
            "Chi-square with 10,000 permutations"
        ),
        "minimum_expected": minimum_expected,
        "percentage_at_least_five": (
            percentage_expected_at_least_five
        )
    }


categorical_table_rows = []
categorical_test_records = []


for variable, label in (
    categorical_variable_labels.items()
):

    test_result = select_categorical_test(
        data=analysis_data,
        predictor=variable
    )

    # Variable heading row
    categorical_table_rows.append({
        "Characteristic": label,
        "No recurrence (N=1,297)": "",
        "Recurrence (N=73)": "",
        "Overall (N=1,370)": "",
        "P value": format_p_value(
            test_result["p_value"]
        )
    })

    # Category rows
    for level in category_orders[variable]:

        no_count = int(
            no_recurrence_data[variable]
            .eq(level)
            .sum()
        )

        recurrence_count = int(
            recurrence_data[variable]
            .eq(level)
            .sum()
        )

        overall_count = int(
            analysis_data[variable]
            .eq(level)
            .sum()
        )

        displayed_level = (
            category_display_labels.get(
                level,
                level.replace("_", " ")
            )
        )

        categorical_table_rows.append({
            "Characteristic": (
                f"    {displayed_level}"
            ),
            "No recurrence (N=1,297)": (
                format_count_percentage(
                    no_count,
                    len(no_recurrence_data)
                )
            ),
            "Recurrence (N=73)": (
                format_count_percentage(
                    recurrence_count,
                    len(recurrence_data)
                )
            ),
            "Overall (N=1,370)": (
                format_count_percentage(
                    overall_count,
                    len(analysis_data)
                )
            ),
            "P value": ""
        })

    categorical_test_records.append({
        "Variable": label,
        "Test": test_result["test"],
        "Raw p value": test_result["p_value"],
        "Minimum expected count": round(
            test_result["minimum_expected"],
            3
        ),
        "Cells with expected count >=5 (%)": round(
            test_result[
                "percentage_at_least_five"
            ],
            1
        )
    })


categorical_table1 = pd.DataFrame(
    categorical_table_rows
)

categorical_test_summary = pd.DataFrame(
    categorical_test_records
)


print("TABLE 1: CATEGORICAL VARIABLES")

display(categorical_table1)


print("\nCATEGORICAL TEST DETAILS")

display(categorical_test_summary)

In [ ]:
continuous_table_section = pd.DataFrame(continuous_table_rows)
categorical_table_section = pd.DataFrame(categorical_table_rows)

table1 = pd.concat(
    [
        continuous_table_section,
        categorical_table_section
    ],
    ignore_index=True
)

expected_table1_columns = [
    "Characteristic",
    "No recurrence (N=1,297)",
    "Recurrence (N=73)",
    "Overall (N=1,370)",
    "P value"
]

assert table1.columns.tolist() == expected_table1_columns
assert len(table1) > 0
assert table1["Characteristic"].notna().all()

print(
    "TABLE 1. CHARACTERISTICS OF PATIENTS "
    "BY RECURRENCE STATUS"
)

display(table1)

print("Number of rows in Table 1:", len(table1))

In [ ]:
# ============================================================
# Finalize and validate Table 1
# ============================================================

final_table1 = table1.copy()


# Apply American English consistently.
final_table1["Characteristic"] = (
    final_table1["Characteristic"]
    .str.replace(
        "Tumour",
        "Tumor",
        regex=False
    )
    .str.replace(
        "tumour",
        "tumor",
        regex=False
    )
)


expected_table_columns = [
    "Characteristic",
    "No recurrence (N=1,297)",
    "Recurrence (N=73)",
    "Overall (N=1,370)",
    "P value"
]


assert (
    final_table1.columns.tolist()
    == expected_table_columns
), "Unexpected Table 1 columns."

assert len(final_table1) == 68, (
    "Expected 68 rows in Table 1."
)

assert (
    final_table1["Characteristic"]
    .notna()
    .all()
), "A characteristic label is missing."

assert (
    final_table1["Characteristic"]
    .str.contains(
        "Tumour|tumour",
        regex=True
    )
    .sum()
    == 0
), "British spelling remains in Table 1."


print("Final Table 1 validation passed.")

display(final_table1)

In [ ]:
# ============================================================
# Export the formatted Excel version of Table 1
# ============================================================

from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side
)


TABLE1_EXCEL_FILE = (
    TABLE_DIRECTORY
    / "table_1_cohort_characteristics.xlsx"
)


with pd.ExcelWriter(
    TABLE1_EXCEL_FILE,
    engine="openpyxl"
) as writer:

    final_table1.to_excel(
        writer,
        sheet_name="Table 1",
        index=False,
        startrow=2
    )

    continuous_test_summary.to_excel(
        writer,
        sheet_name="Continuous Tests",
        index=False
    )

    categorical_test_summary.to_excel(
        writer,
        sheet_name="Categorical Tests",
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets["Table 1"]

    # Table title
    worksheet["A1"] = (
        "Table 1. Characteristics of patients "
        "by recurrence status"
    )

    worksheet.merge_cells(
        start_row=1,
        start_column=1,
        end_row=1,
        end_column=5
    )

    worksheet["A1"].font = Font(
        bold=True,
        size=14,
        color="FFFFFF"
    )

    worksheet["A1"].fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    worksheet["A1"].alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

    worksheet.row_dimensions[1].height = 25

    # Column headers
    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )

    for cell in worksheet[3]:

        cell.font = Font(
            bold=True,
            color="000000"
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    worksheet.row_dimensions[3].height = 35

    # Border style
    thin_border = Border(
        bottom=Side(
            style="thin",
            color="B7B7B7"
        )
    )

    section_fill = PatternFill(
        fill_type="solid",
        fgColor="EDEDED"
    )

    # Identify categorical variable heading rows.
    categorical_headings = set(
        categorical_variable_labels.values()
    )

    categorical_headings = {
        heading.replace("Tumour", "Tumor")
        for heading in categorical_headings
    }

    for row_number in range(
        4,
        worksheet.max_row + 1
    ):

        characteristic = worksheet.cell(
            row=row_number,
            column=1
        ).value

        if characteristic in categorical_headings:

            for column_number in range(1, 6):

                cell = worksheet.cell(
                    row=row_number,
                    column=column_number
                )

                cell.font = Font(bold=True)
                cell.fill = section_fill
                cell.border = thin_border

        worksheet.cell(
            row=row_number,
            column=1
        ).alignment = Alignment(
            horizontal="left",
            vertical="center",
            wrap_text=True
        )

        for column_number in range(2, 6):

            worksheet.cell(
                row=row_number,
                column=column_number
            ).alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

    # Column widths
    worksheet.column_dimensions["A"].width = 42
    worksheet.column_dimensions["B"].width = 25
    worksheet.column_dimensions["C"].width = 22
    worksheet.column_dimensions["D"].width = 22
    worksheet.column_dimensions["E"].width = 12

    worksheet.freeze_panes = "A4"

    # Footnotes
    footnote_row = worksheet.max_row + 3

    footnotes = [
        (
            "Values are n (%) unless otherwise stated. "
            "Percentages were calculated using the total "
            "number of patients in the corresponding column."
        ),
        (
            "Continuous variables are presented as median "
            "[interquartile range] and were compared using "
            "the two-sided Mann-Whitney U test."
        ),
        (
            "Categorical variables were compared using "
            "Pearson's chi-square test, Fisher's exact test, "
            "or a chi-square permutation test with 10,000 "
            "permutations, as appropriate."
        ),
        (
            "Unknown values were retained as explicit "
            "categories and included in the primary "
            "categorical comparisons."
        ),
        (
            "Tumor size was observed for 1,196 patients and "
            "missing for 174 patients. No imputation was "
            "performed for the descriptive analysis."
        )
    ]

    for offset, footnote in enumerate(footnotes):

        current_row = footnote_row + offset

        worksheet.merge_cells(
            start_row=current_row,
            start_column=1,
            end_row=current_row,
            end_column=5
        )

        cell = worksheet.cell(
            row=current_row,
            column=1,
            value=footnote
        )

        cell.font = Font(
            italic=True,
            size=10
        )

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )

        worksheet.row_dimensions[
            current_row
        ].height = 30


print("Formatted Excel Table 1 saved successfully:")
print(TABLE1_EXCEL_FILE)

In [ ]:
# ============================================================
# Export Table 1 as an Overleaf-ready LaTeX longtable
# ============================================================

TABLE1_LATEX_FILE = (
    TABLE_DIRECTORY
    / "table1_characteristics.tex"
)


def escape_latex(value):
    """
    Escape text that has special meaning in LaTeX.
    """

    if pd.isna(value):
        return ""

    text = str(value)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}"
    }

    for original, replacement in replacements.items():
        text = text.replace(
            original,
            replacement
        )

    return text


latex_lines = [
    r"\begingroup",
    r"\small",
    r"\setlength{\tabcolsep}{4pt}",
    r"\renewcommand{\arraystretch}{1.12}",
    r"\begin{longtable}{"
    r"p{4.5cm} "
    r">{\centering\arraybackslash}p{2.6cm} "
    r">{\centering\arraybackslash}p{2.4cm} "
    r">{\centering\arraybackslash}p{2.5cm} "
    r">{\centering\arraybackslash}p{1.2cm}}",
    (
        r"\caption{Characteristics of patients "
        r"by recurrence status}"
        r"\label{tab:characteristics}\\"
    ),
    r"\toprule",
    (
        r"\textbf{Characteristic} & "
        r"\textbf{No recurrence} & "
        r"\textbf{Recurrence} & "
        r"\textbf{Overall} & "
        r"\textbf{\(p\) value} \\"
    ),
    (
        r" & "
        r"\textbf{(\(N=1{,}297\))} & "
        r"\textbf{(\(N=73\))} & "
        r"\textbf{(\(N=1{,}370\))} & \\"
    ),
    r"\midrule",
    r"\endfirsthead",
    (
        r"\multicolumn{5}{c}{"
        r"\tablename\ \thetable\ continued} \\"
    ),
    r"\toprule",
    (
        r"\textbf{Characteristic} & "
        r"\textbf{No recurrence} & "
        r"\textbf{Recurrence} & "
        r"\textbf{Overall} & "
        r"\textbf{\(p\) value} \\"
    ),
    r"\midrule",
    r"\endhead",
    r"\midrule",
    (
        r"\multicolumn{5}{r}{"
        r"\textit{Continued on the next page}} \\"
    ),
    r"\endfoot",
    r"\bottomrule",
    r"\endlastfoot"
]


for _, row in final_table1.iterrows():

    characteristic = str(
        row["Characteristic"]
    )

    no_result = str(
        row["No recurrence (N=1,297)"]
    )

    recurrence_result = str(
        row["Recurrence (N=73)"]
    )

    overall_result = str(
        row["Overall (N=1,370)"]
    )

    p_value = str(
        row["P value"]
    )

    is_categorical_heading = (
        no_result == ""
        and recurrence_result == ""
        and overall_result == ""
    )

    if is_categorical_heading:

        latex_lines.append(r"\addlinespace[2pt]")

        latex_lines.append(
            r"\textbf{"
            + escape_latex(characteristic.strip())
            + r"} & & & & "
            + escape_latex(p_value)
            + r" \\"
        )

    else:

        is_indented_category = (
            characteristic.startswith("    ")
        )

        if is_indented_category:
            displayed_characteristic = (
                r"\hspace{1em}"
                + escape_latex(
                    characteristic.strip()
                )
            )
        else:
            displayed_characteristic = (
                escape_latex(
                    characteristic.strip()
                )
            )

        latex_lines.append(
            displayed_characteristic
            + " & "
            + escape_latex(no_result)
            + " & "
            + escape_latex(recurrence_result)
            + " & "
            + escape_latex(overall_result)
            + " & "
            + escape_latex(p_value)
            + r" \\"
        )


latex_lines.extend([
    r"\end{longtable}",
    r"\begin{minipage}{\textwidth}",
    r"\footnotesize",
    (
        r"\textit{Notes:} Values are \(n\) (\%) unless "
        r"otherwise stated. Percentages were calculated using "
        r"the total number of patients in the corresponding "
        r"column. Continuous variables are presented as median "
        r"[interquartile range] and were compared using the "
        r"two-sided Mann--Whitney \(U\) test. Categorical "
        r"variables were compared using Pearson's chi-square "
        r"test, Fisher's exact test, or a chi-square permutation "
        r"test with 10,000 permutations, as appropriate. "
        r"Unknown values were retained as explicit categories. "
        r"Tumor size was observed for 1,196 patients and missing "
        r"for 174 patients; no imputation was performed for the "
        r"descriptive analysis."
    ),
    r"\end{minipage}",
    r"\endgroup"
])


TABLE1_LATEX_FILE.write_text(
    "\n".join(latex_lines),
    encoding="utf-8"
)


assert TABLE1_LATEX_FILE.exists()

assert TABLE1_LATEX_FILE.stat().st_size > 0


print("LaTeX Table 1 saved successfully:")
print(TABLE1_LATEX_FILE)

## 2. Missing and Unknown Information

In [ ]:
# ============================================================
# Missing and unknown information summary
# ============================================================

OUTCOME_COLUMN = "Recurrence"


predictor_columns = [
    column
    for column in analysis_data.columns
    if column != OUTCOME_COLUMN
]


variable_display_labels = {
    "Age_at_Diagnosis": "Age at diagnosis",
    "Marital_Status": "Marital status",
    "Nationality": "Nationality",
    "Menopause_Status": "Menopause status",
    "Breast_Cancer_Family_History": (
        "Breast cancer family history"
    ),
    "Primary_Site_Group": "Primary tumor site",
    "Laterality": "Laterality",
    "Histology_Group": "Histology",
    "Tumor_Grade": "Tumor grade",
    "Tumor_Size_mm": "Tumor size",
    "Summary_Stage": "Summary stage",
    "ER_Status": "Estrogen receptor status",
    "PR_Status": "Progesterone receptor status",
    "HER2_Status": "HER2 status",
    "Surgery": "Surgery",
    "Chemotherapy": "Chemotherapy",
    "Hormone_Therapy": "Hormone therapy"
}


def identify_unavailable_values(data, variable):
    """
    Identify unavailable information.

    Tumor size uses standard missing values.
    Categorical variables use either standard missing values
    or the explicit category Unknown.
    """

    if variable == "Tumor_Size_mm":

        unavailable_mask = data[variable].isna()
        unavailable_type = "Missing"

    else:

        unavailable_mask = (
            data[variable].isna()
            |
            data[variable]
            .astype("string")
            .str.strip()
            .eq("Unknown")
        )

        unavailable_type = "Unknown"

    return unavailable_mask, unavailable_type


missingness_rows = []
missingness_test_rows = []


for variable in predictor_columns:

    unavailable_mask, unavailable_type = (
        identify_unavailable_values(
            analysis_data,
            variable
        )
    )

    overall_unavailable = int(
        unavailable_mask.sum()
    )

    no_recurrence_mask = (
        analysis_data[OUTCOME_COLUMN]
        .eq("No")
    )

    recurrence_mask = (
        analysis_data[OUTCOME_COLUMN]
        .eq("Yes")
    )

    no_recurrence_unavailable = int(
        (
            unavailable_mask
            & no_recurrence_mask
        ).sum()
    )

    recurrence_unavailable = int(
        (
            unavailable_mask
            & recurrence_mask
        ).sum()
    )

    missingness_rows.append({
        "Variable": variable_display_labels[variable],
        "Unavailable type": unavailable_type,
        "Overall n": overall_unavailable,
        "Overall %": round(
            overall_unavailable
            / len(analysis_data)
            * 100,
            2
        ),
        "No recurrence n": no_recurrence_unavailable,
        "No recurrence %": round(
            no_recurrence_unavailable
            / no_recurrence_mask.sum()
            * 100,
            2
        ),
        "Recurrence n": recurrence_unavailable,
        "Recurrence %": round(
            recurrence_unavailable
            / recurrence_mask.sum()
            * 100,
            2
        )
    })

    # Compare availability with recurrence only when
    # unavailable information is present.
    if 0 < overall_unavailable < len(analysis_data):

        availability_table = pd.crosstab(
            unavailable_mask.map({
                False: "Available",
                True: "Unavailable"
            }),
            analysis_data[OUTCOME_COLUMN]
        )

        if availability_table.shape == (2, 2):

            chi_square_result = chi2_contingency(
                availability_table,
                correction=False
            )

            expected_counts = (
                chi_square_result.expected_freq
            )

            if expected_counts.min() >= 5:

                p_value = chi_square_result.pvalue
                statistical_test = "Pearson chi-square"

            else:

                fisher_result = fisher_exact(
                    availability_table.to_numpy()
                )

                p_value = fisher_result.pvalue
                statistical_test = "Fisher's exact"

            missingness_test_rows.append({
                "Variable": variable_display_labels[variable],
                "Test": statistical_test,
                "Raw p value": p_value,
                "Formatted p value": (
                    format_p_value(p_value)
                ),
                "Minimum expected count": round(
                    expected_counts.min(),
                    3
                )
            })


missingness_summary = (
    pd.DataFrame(missingness_rows)
    .sort_values(
        by=[
            "Overall %",
            "Variable"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


missingness_test_summary = (
    pd.DataFrame(missingness_test_rows)
    .sort_values(
        by="Raw p value"
    )
    .reset_index(drop=True)
)


print("MISSING AND UNKNOWN INFORMATION")

display(missingness_summary)


print(
    "\nASSOCIATION BETWEEN INFORMATION AVAILABILITY "
    "AND RECURRENCE"
)

display(missingness_test_summary)

In [ ]:
# ============================================================
# Examine overlapping unavailable information
# ============================================================

key_availability_variables = [
    "Tumor_Grade",
    "Tumor_Size_mm",
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]


availability_indicators = pd.DataFrame(
    index=analysis_data.index
)


for variable in key_availability_variables:

    unavailable_mask, _ = identify_unavailable_values(
        analysis_data,
        variable
    )

    availability_indicators[
        variable
    ] = unavailable_mask.astype(int)


availability_indicators[
    "Recurrence"
] = analysis_data[
    OUTCOME_COLUMN
]


# ------------------------------------------------------------
# Pairwise overlap counts
# ------------------------------------------------------------

unavailable_overlap_counts = (
    availability_indicators[
        key_availability_variables
    ]
    .T
    .dot(
        availability_indicators[
            key_availability_variables
        ]
    )
)


unavailable_overlap_counts.index = [
    variable_display_labels[variable]
    for variable in unavailable_overlap_counts.index
]

unavailable_overlap_counts.columns = [
    variable_display_labels[variable]
    for variable in unavailable_overlap_counts.columns
]


print("PAIRWISE COUNTS OF UNAVAILABLE INFORMATION")

display(unavailable_overlap_counts)


# ------------------------------------------------------------
# Create patient-level missingness patterns
# ------------------------------------------------------------

short_names = {
    "Tumor_Grade": "Grade",
    "Tumor_Size_mm": "Tumor size",
    "ER_Status": "ER",
    "PR_Status": "PR",
    "HER2_Status": "HER2"
}


def describe_unavailable_pattern(row):

    unavailable_items = [
        short_names[variable]
        for variable in key_availability_variables
        if row[variable] == 1
    ]

    if not unavailable_items:
        return "All selected variables available"

    return "Unavailable: " + ", ".join(
        unavailable_items
    )


availability_indicators[
    "Availability_Pattern"
] = (
    availability_indicators.apply(
        describe_unavailable_pattern,
        axis=1
    )
)


pattern_by_outcome = pd.crosstab(
    availability_indicators[
        "Availability_Pattern"
    ],
    availability_indicators[
        "Recurrence"
    ]
)


# Ensure that both outcome columns exist.
for outcome_level in ["No", "Yes"]:

    if outcome_level not in pattern_by_outcome.columns:
        pattern_by_outcome[outcome_level] = 0


pattern_by_outcome = pattern_by_outcome[
    [
        "No",
        "Yes"
    ]
]


pattern_by_outcome["Overall"] = (
    pattern_by_outcome["No"]
    + pattern_by_outcome["Yes"]
)


pattern_by_outcome[
    "Recurrence_Percentage"
] = (
    pattern_by_outcome["Yes"]
    / pattern_by_outcome["Overall"]
    * 100
).round(2)


pattern_by_outcome = (
    pattern_by_outcome
    .sort_values(
        by="Overall",
        ascending=False
    )
    .reset_index()
)


print("\nUNAVAILABLE-INFORMATION PATTERNS")

display(pattern_by_outcome)


# ------------------------------------------------------------
# Summarize receptor availability
# ------------------------------------------------------------

receptor_variables = [
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]


receptor_unavailable_count = (
    availability_indicators[
        receptor_variables
    ]
    .sum(axis=1)
)


receptor_availability_group = np.select(
    [
        receptor_unavailable_count.eq(0),
        receptor_unavailable_count.eq(3)
    ],
    [
        "All three available",
        "All three unavailable"
    ],
    default="Partially available"
)


receptor_availability_summary = pd.crosstab(
    receptor_availability_group,
    analysis_data[OUTCOME_COLUMN],
    margins=True
)


print("\nER, PR, AND HER2 AVAILABILITY SUMMARY")

display(receptor_availability_summary)

In [ ]:
# ============================================================
# Load the auditable cleaned cohort for temporal analysis
# ============================================================

CLEANED_COHORT_FILE = (
    PROCESSED_DATA_DIRECTORY
    / "cleaned_analysis_cohort.xlsx"
)


if not CLEANED_COHORT_FILE.exists():
    raise FileNotFoundError(
        "The cleaned auditable cohort was not found:\n"
        f"{CLEANED_COHORT_FILE}"
    )


audit_data = pd.read_excel(
    CLEANED_COHORT_FILE
)


assert len(audit_data) == 1370, (
    "Expected 1,370 patients in the cleaned cohort."
)


date_related_columns = [
    column
    for column in audit_data.columns
    if (
        "date" in column.lower()
        or "year" in column.lower()
    )
]


receptor_related_columns = [
    column
    for column in audit_data.columns
    if any(
        term in column.lower()
        for term in [
            "estrogen",
            "progesterone",
            "her2",
            "er_status",
            "pr_status"
        ]
    )
]


print("AUDIT DATASET LOADED")
print("--------------------")
print("Patients:", audit_data.shape[0])
print("Variables:", audit_data.shape[1])


print("\nDate- and year-related columns:")

for column in date_related_columns:
    print("-", repr(column))


print("\nReceptor-related columns:")

for column in receptor_related_columns:
    print("-", repr(column))

In [ ]:
# ============================================================
# Temporal distribution of unavailable clinical information
# ============================================================

required_audit_columns = [
    "Diagnosis_Date_Used",
    "Five_Year_Recurrence",
    "Combined_Grade",
    "Tumor_Size_mm",
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]


missing_audit_columns = [
    column
    for column in required_audit_columns
    if column not in audit_data.columns
]


if missing_audit_columns:
    raise KeyError(
        "Required audit variables are missing:\n"
        + "\n".join(missing_audit_columns)
    )


temporal_data = audit_data[
    required_audit_columns
].copy()


temporal_data["Diagnosis_Date_Used"] = pd.to_datetime(
    temporal_data["Diagnosis_Date_Used"],
    errors="coerce"
)


temporal_data["Diagnosis_Year"] = (
    temporal_data["Diagnosis_Date_Used"]
    .dt.year
)


assert (
    temporal_data["Diagnosis_Year"]
    .notna()
    .all()
), "At least one diagnosis year is unavailable."


# ------------------------------------------------------------
# Create unavailable-information indicators
# ------------------------------------------------------------

temporal_data["Grade_Unknown"] = (
    temporal_data["Combined_Grade"]
    .eq("Unknown")
)


temporal_data["ER_Unknown"] = (
    temporal_data["ER_Status"]
    .eq("Unknown")
)


temporal_data["PR_Unknown"] = (
    temporal_data["PR_Status"]
    .eq("Unknown")
)


temporal_data["HER2_Unknown"] = (
    temporal_data["HER2_Status"]
    .eq("Unknown")
)


temporal_data["All_Receptors_Unknown"] = (
    temporal_data[
        [
            "ER_Unknown",
            "PR_Unknown",
            "HER2_Unknown"
        ]
    ]
    .all(axis=1)
)


temporal_data["All_Receptors_Available"] = (
    ~temporal_data[
        [
            "ER_Unknown",
            "PR_Unknown",
            "HER2_Unknown"
        ]
    ]
    .any(axis=1)
)


temporal_data["Tumor_Size_Missing"] = (
    temporal_data["Tumor_Size_mm"]
    .isna()
)


temporal_data["Recurrence_Event"] = (
    temporal_data["Five_Year_Recurrence"]
    .eq("Yes")
)


# ------------------------------------------------------------
# Summarize by diagnosis year
# ------------------------------------------------------------

temporal_missingness_summary = (
    temporal_data
    .groupby(
        "Diagnosis_Year",
        observed=True
    )
    .agg(
        Patients=(
            "Diagnosis_Year",
            "size"
        ),
        Recurrences=(
            "Recurrence_Event",
            "sum"
        ),
        Grade_Unknown=(
            "Grade_Unknown",
            "sum"
        ),
        All_Receptors_Unknown=(
            "All_Receptors_Unknown",
            "sum"
        ),
        All_Receptors_Available=(
            "All_Receptors_Available",
            "sum"
        ),
        Tumor_Size_Missing=(
            "Tumor_Size_Missing",
            "sum"
        )
    )
    .reset_index()
)


temporal_missingness_summary[
    "Recurrence_Percentage"
] = (
    temporal_missingness_summary["Recurrences"]
    / temporal_missingness_summary["Patients"]
    * 100
).round(2)


temporal_missingness_summary[
    "Grade_Unknown_Percentage"
] = (
    temporal_missingness_summary["Grade_Unknown"]
    / temporal_missingness_summary["Patients"]
    * 100
).round(2)


temporal_missingness_summary[
    "All_Receptors_Unknown_Percentage"
] = (
    temporal_missingness_summary[
        "All_Receptors_Unknown"
    ]
    / temporal_missingness_summary["Patients"]
    * 100
).round(2)


temporal_missingness_summary[
    "All_Receptors_Available_Percentage"
] = (
    temporal_missingness_summary[
        "All_Receptors_Available"
    ]
    / temporal_missingness_summary["Patients"]
    * 100
).round(2)


temporal_missingness_summary[
    "Tumor_Size_Missing_Percentage"
] = (
    temporal_missingness_summary[
        "Tumor_Size_Missing"
    ]
    / temporal_missingness_summary["Patients"]
    * 100
).round(2)


print("TEMPORAL MISSINGNESS BY DIAGNOSIS YEAR")

display(
    temporal_missingness_summary[
        [
            "Diagnosis_Year",
            "Patients",
            "Recurrences",
            "Recurrence_Percentage",
            "Grade_Unknown_Percentage",
            "All_Receptors_Unknown_Percentage",
            "All_Receptors_Available_Percentage",
            "Tumor_Size_Missing_Percentage"
        ]
    ]
)


print(
    "Diagnosis-year range:",
    int(
        temporal_data[
            "Diagnosis_Year"
        ].min()
    ),
    "to",
    int(
        temporal_data[
            "Diagnosis_Year"
        ].max()
    )
)

In [ ]:
# ============================================================
# Summarize missingness by clinically meaningful time periods
# ============================================================

temporal_data["Diagnosis_Period"] = np.select(
    [
        temporal_data["Diagnosis_Year"].le(2017),
        temporal_data["Diagnosis_Year"].eq(2018),
        temporal_data["Diagnosis_Year"].eq(2019)
    ],
    [
        "2002-2017",
        "2018",
        "2019"
    ],
    default="Other"
)


period_order = [
    "2002-2017",
    "2018",
    "2019"
]


temporal_data["Diagnosis_Period"] = pd.Categorical(
    temporal_data["Diagnosis_Period"],
    categories=period_order,
    ordered=True
)


period_missingness_summary = (
    temporal_data
    .groupby(
        "Diagnosis_Period",
        observed=True
    )
    .agg(
        Patients=(
            "Diagnosis_Period",
            "size"
        ),
        Recurrences=(
            "Recurrence_Event",
            "sum"
        ),
        Grade_Unknown=(
            "Grade_Unknown",
            "sum"
        ),
        All_Receptors_Unknown=(
            "All_Receptors_Unknown",
            "sum"
        ),
        Tumor_Size_Missing=(
            "Tumor_Size_Missing",
            "sum"
        )
    )
    .reset_index()
)


period_missingness_summary[
    "Recurrence_Percentage"
] = (
    period_missingness_summary["Recurrences"]
    / period_missingness_summary["Patients"]
    * 100
).round(2)


period_missingness_summary[
    "Grade_Unknown_Percentage"
] = (
    period_missingness_summary["Grade_Unknown"]
    / period_missingness_summary["Patients"]
    * 100
).round(2)


period_missingness_summary[
    "Receptors_Unknown_Percentage"
] = (
    period_missingness_summary[
        "All_Receptors_Unknown"
    ]
    / period_missingness_summary["Patients"]
    * 100
).round(2)


period_missingness_summary[
    "Tumor_Size_Missing_Percentage"
] = (
    period_missingness_summary[
        "Tumor_Size_Missing"
    ]
    / period_missingness_summary["Patients"]
    * 100
).round(2)


print("MISSINGNESS AND RECURRENCE BY DIAGNOSIS PERIOD")

display(period_missingness_summary)

In [ ]:
# ============================================================
# Plot missingness and recurrence by diagnosis period
# ============================================================

plt.close("all")


plot_data = period_missingness_summary.copy()


diagnosis_periods = (
    plot_data["Diagnosis_Period"]
    .astype(str)
    .tolist()
)


x_positions = np.arange(
    len(diagnosis_periods)
)


patient_counts = (
    plot_data["Patients"]
    .to_numpy(dtype=float)
)

recurrence_percentages = (
    plot_data["Recurrence_Percentage"]
    .to_numpy(dtype=float)
)

grade_unknown_percentages = (
    plot_data["Grade_Unknown_Percentage"]
    .to_numpy(dtype=float)
)

receptor_unknown_percentages = (
    plot_data["Receptors_Unknown_Percentage"]
    .to_numpy(dtype=float)
)

tumor_size_missing_percentages = (
    plot_data["Tumor_Size_Missing_Percentage"]
    .to_numpy(dtype=float)
)


fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5)
)


# ------------------------------------------------------------
# Panel A: Cohort size and recurrence percentage
# ------------------------------------------------------------

patient_bars = axes[0].bar(
    x_positions,
    patient_counts,
    color="#2F75B5",
    edgecolor="black",
    linewidth=0.6
)


axes[0].set_title(
    "A. Cohort and Recurrence by Period",
    fontweight="bold"
)

axes[0].set_xlabel("Diagnosis period")
axes[0].set_ylabel("Number of patients")

axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(diagnosis_periods)


for bar, patient_count in zip(
    patient_bars,
    patient_counts
):

    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"{int(patient_count):,}",
        ha="center",
        va="bottom"
    )


recurrence_axis = axes[0].twinx()


recurrence_axis.plot(
    x_positions,
    recurrence_percentages,
    color="#C00000",
    marker="o",
    markersize=7,
    linewidth=2
)


recurrence_axis.set_ylabel(
    "Recorded recurrence (%)",
    color="#C00000"
)

recurrence_axis.tick_params(
    axis="y",
    colors="#C00000"
)

recurrence_axis.set_ylim(
    0,
    max(
        10,
        recurrence_percentages.max() + 2
    )
)


# Add recurrence percentages above the markers.
for position, percentage in zip(
    x_positions,
    recurrence_percentages
):

    recurrence_axis.text(
        position,
        percentage + 0.4,
        f"{percentage:.2f}%",
        color="#C00000",
        ha="center",
        va="bottom"
    )


# ------------------------------------------------------------
# Panel B: Unavailable predictor information
# ------------------------------------------------------------

axes[1].plot(
    x_positions,
    grade_unknown_percentages,
    marker="o",
    linewidth=2,
    label="Tumor grade unknown"
)


axes[1].plot(
    x_positions,
    receptor_unknown_percentages,
    marker="s",
    linewidth=2,
    label="ER, PR, and HER2 unknown"
)


axes[1].plot(
    x_positions,
    tumor_size_missing_percentages,
    marker="^",
    linewidth=2,
    label="Tumor size missing"
)


axes[1].set_title(
    "B. Unavailable Predictor Information",
    fontweight="bold"
)

axes[1].set_xlabel("Diagnosis period")

axes[1].set_ylabel(
    "Patients with unavailable information (%)"
)

axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(diagnosis_periods)

axes[1].set_ylim(0, 105)

axes[1].legend(
    loc="best",
    frameon=True
)


# ------------------------------------------------------------
# Final formatting and export
# ------------------------------------------------------------

axes[0].spines["top"].set_visible(False)
axes[1].spines["top"].set_visible(False)
recurrence_axis.spines["top"].set_visible(False)


fig.tight_layout()


TEMPORAL_MISSINGNESS_PNG = (
    FIGURE_DIRECTORY
    / "temporal_missingness_by_diagnosis_period.png"
)

TEMPORAL_MISSINGNESS_PDF = (
    FIGURE_DIRECTORY
    / "temporal_missingness_by_diagnosis_period.pdf"
)


fig.savefig(
    TEMPORAL_MISSINGNESS_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    TEMPORAL_MISSINGNESS_PDF,
    bbox_inches="tight"
)


plt.show()


print("Temporal missingness figures saved:")
print(TEMPORAL_MISSINGNESS_PNG)
print(TEMPORAL_MISSINGNESS_PDF)

In [ ]:
# ============================================================
# Export missingness-analysis results
# ============================================================

MISSINGNESS_RESULTS_FILE = (
    TABLE_DIRECTORY
    / "missing_information_analysis.xlsx"
)


with pd.ExcelWriter(
    MISSINGNESS_RESULTS_FILE,
    engine="openpyxl"
) as writer:

    missingness_summary.to_excel(
        writer,
        sheet_name="Missingness Summary",
        index=False
    )

    missingness_test_summary.to_excel(
        writer,
        sheet_name="Availability Tests",
        index=False
    )

    unavailable_overlap_counts.to_excel(
        writer,
        sheet_name="Pairwise Overlap"
    )

    pattern_by_outcome.to_excel(
        writer,
        sheet_name="Missingness Patterns",
        index=False
    )

    receptor_availability_summary.to_excel(
        writer,
        sheet_name="Receptor Availability"
    )

    temporal_missingness_summary.to_excel(
        writer,
        sheet_name="Yearly Pattern",
        index=False
    )

    period_missingness_summary.to_excel(
        writer,
        sheet_name="Period Pattern",
        index=False
    )


print("Missingness-analysis workbook saved:")
print(MISSINGNESS_RESULTS_FILE)

## 3. Predictor Relationships and Redundancy

In [ ]:
# ============================================================
# Cramer's V associations among categorical predictors
# ============================================================

from itertools import combinations


categorical_predictors = [
    "Marital_Status",
    "Nationality",
    "Menopause_Status",
    "Breast_Cancer_Family_History",
    "Primary_Site_Group",
    "Laterality",
    "Histology_Group",
    "Tumor_Grade",
    "Summary_Stage",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Surgery",
    "Chemotherapy",
    "Hormone_Therapy"
]


def corrected_cramers_v(first_variable, second_variable):
    """
    Calculate bias-corrected Cramer's V for two
    categorical variables.
    """

    contingency_table = pd.crosstab(
        first_variable,
        second_variable
    )

    chi_square = chi2_contingency(
        contingency_table,
        correction=False
    ).statistic

    sample_size = contingency_table.to_numpy().sum()

    number_of_rows = contingency_table.shape[0]
    number_of_columns = contingency_table.shape[1]

    phi_squared = chi_square / sample_size

    corrected_phi_squared = max(
        0,
        phi_squared
        - (
            (number_of_columns - 1)
            * (number_of_rows - 1)
            / (sample_size - 1)
        )
    )

    corrected_rows = (
        number_of_rows
        - (
            (number_of_rows - 1) ** 2
            / (sample_size - 1)
        )
    )

    corrected_columns = (
        number_of_columns
        - (
            (number_of_columns - 1) ** 2
            / (sample_size - 1)
        )
    )

    denominator = min(
        corrected_rows - 1,
        corrected_columns - 1
    )

    if denominator <= 0:
        return np.nan

    return np.sqrt(
        corrected_phi_squared
        / denominator
    )


# ------------------------------------------------------------
# Calculate every predictor pair
# ------------------------------------------------------------

cramers_v_records = []


for first_variable, second_variable in combinations(
    categorical_predictors,
    2
):

    association = corrected_cramers_v(
        analysis_data[first_variable],
        analysis_data[second_variable]
    )

    cramers_v_records.append({
        "Predictor 1": (
            variable_display_labels[first_variable]
        ),
        "Predictor 2": (
            variable_display_labels[second_variable]
        ),
        "Cramer's V": round(
            association,
            3
        )
    })


categorical_association_pairs = (
    pd.DataFrame(cramers_v_records)
    .sort_values(
        by="Cramer's V",
        ascending=False
    )
    .reset_index(drop=True)
)


def classify_association(value):

    if pd.isna(value):
        return "Not available"

    if value < 0.10:
        return "Negligible"

    if value < 0.30:
        return "Weak"

    if value < 0.50:
        return "Moderate"

    return "Strong"


categorical_association_pairs[
    "Interpretation"
] = (
    categorical_association_pairs[
        "Cramer's V"
    ]
    .apply(classify_association)
)


print("STRONGEST CATEGORICAL PREDICTOR RELATIONSHIPS")

display(
    categorical_association_pairs.head(20)
)


print("\nRELATIONSHIPS WITH CRAMER'S V OF AT LEAST 0.30")

display(
    categorical_association_pairs.loc[
        categorical_association_pairs[
            "Cramer's V"
        ].ge(0.30)
    ]
)

In [ ]:
# ============================================================
# Recalculate categorical relationships among known values
# ============================================================

known_value_association_records = []


for first_variable, second_variable in combinations(
    categorical_predictors,
    2
):

    first_values = analysis_data[
        first_variable
    ]

    second_values = analysis_data[
        second_variable
    ]


    known_value_mask = (
        first_values.notna()
        &
        second_values.notna()
        &
        first_values.astype("string").ne("Unknown")
        &
        second_values.astype("string").ne("Unknown")
    )


    known_first = first_values.loc[
        known_value_mask
    ]

    known_second = second_values.loc[
        known_value_mask
    ]


    if (
        known_first.nunique() < 2
        or known_second.nunique() < 2
    ):

        known_association = np.nan

    else:

        known_association = corrected_cramers_v(
            known_first,
            known_second
        )


    all_value_association = corrected_cramers_v(
        first_values,
        second_values
    )


    known_value_association_records.append({
        "Predictor 1": (
            variable_display_labels[
                first_variable
            ]
        ),
        "Predictor 2": (
            variable_display_labels[
                second_variable
            ]
        ),
        "Complete paired observations": int(
            known_value_mask.sum()
        ),
        "Cramer's V including Unknown": round(
            all_value_association,
            3
        ),
        "Cramer's V among known values": (
            round(
                known_association,
                3
            )
            if pd.notna(known_association)
            else np.nan
        )
    })


known_value_associations = pd.DataFrame(
    known_value_association_records
)


known_value_associations[
    "Change after excluding Unknown"
] = (
    known_value_associations[
        "Cramer's V among known values"
    ]
    -
    known_value_associations[
        "Cramer's V including Unknown"
    ]
).round(3)


known_value_associations = (
    known_value_associations
    .sort_values(
        by="Cramer's V among known values",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "STRONGEST RELATIONSHIPS AMONG KNOWN VALUES"
)

display(
    known_value_associations.head(20)
)


print(
    "\nGRADE AND RECEPTOR RELATIONSHIPS "
    "BEFORE AND AFTER EXCLUDING UNKNOWN"
)

grade_receptor_variables = {
    "Tumor grade",
    "Estrogen receptor status",
    "Progesterone receptor status",
    "HER2 status"
}


grade_receptor_comparison = (
    known_value_associations.loc[
        known_value_associations[
            "Predictor 1"
        ].isin(grade_receptor_variables)
        &
        known_value_associations[
            "Predictor 2"
        ].isin(grade_receptor_variables)
    ]
    .reset_index(drop=True)
)


display(grade_receptor_comparison)

In [ ]:
# ============================================================
# Detailed clinically important categorical relationships
# ============================================================

def display_count_and_row_percentage(
    first_variable,
    second_variable,
    title
):

    count_table = pd.crosstab(
        analysis_data[first_variable],
        analysis_data[second_variable],
        margins=True
    )

    percentage_table = (
        pd.crosstab(
            analysis_data[first_variable],
            analysis_data[second_variable],
            normalize="index"
        )
        * 100
    ).round(1)

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print("\nCounts:")
    display(count_table)

    print("\nRow percentages:")
    display(percentage_table)


# ER versus PR among patients with both results known.
known_er_pr = analysis_data.loc[
    analysis_data["ER_Status"].ne("Unknown")
    &
    analysis_data["PR_Status"].ne("Unknown")
].copy()


print("ER VERSUS PR AMONG KNOWN RESULTS")

display(
    pd.crosstab(
        known_er_pr["ER_Status"],
        known_er_pr["PR_Status"],
        margins=True
    )
)


print("Row percentages:")

display(
    (
        pd.crosstab(
            known_er_pr["ER_Status"],
            known_er_pr["PR_Status"],
            normalize="index"
        )
        * 100
    ).round(1)
)


display_count_and_row_percentage(
    first_variable="ER_Status",
    second_variable="Hormone_Therapy",
    title="ER STATUS VERSUS HORMONE THERAPY"
)


display_count_and_row_percentage(
    first_variable="Summary_Stage",
    second_variable="Surgery",
    title="SUMMARY STAGE VERSUS SURGERY"
)


display_count_and_row_percentage(
    first_variable="Summary_Stage",
    second_variable="Chemotherapy",
    title="SUMMARY STAGE VERSUS CHEMOTHERAPY"
)


display_count_and_row_percentage(
    first_variable="Histology_Group",
    second_variable="Summary_Stage",
    title="HISTOLOGY VERSUS SUMMARY STAGE"
)

In [ ]:
# ============================================================
# Audit histology and summary-stage inconsistencies
# ============================================================

histology_stage_inconsistency = (
    audit_data["Histology_Group"].eq("In_Situ")
    &
    audit_data["Summary_Stage_Clean"].ne("In_Situ")
)


candidate_audit_columns = [
    "Source_Excel_Row",
    "Diagnosis_Date_Used",
    "Histology/Behavior (ICD-O-3)",
    "Histology +
]

In [ ]:
# ============================================================
# Audit in-situ histology with a non-in-situ summary stage
# ============================================================

histology_stage_inconsistency = (
    audit_data["Histology_Group"].eq("In_Situ")
    &
    audit_data["Summary_Stage_Clean"].ne("In_Situ")
)


# Include only columns that actually exist in audit_data.
candidate_audit_columns = [
    "Source_Excel_Row",
    "Diagnosis_Date_Used",
    "Histology/Behavior (ICD-O-3)",
    "Histology_Group",
    "Summary Stage",
    "Summary Stage ",
    "Summary_Stage_Clean",
    "Recurrence",
    "Recurrence ",
    "Five_Year_Recurrence"
]

available_audit_columns = [
    column
    for column in candidate_audit_columns
    if column in audit_data.columns
]


histology_stage_inconsistency_records = (
    audit_data.loc[
        histology_stage_inconsistency,
        available_audit_columns
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "In-situ histology with non-in-situ summary stage:",
    len(histology_stage_inconsistency_records)
)

print("\nColumns displayed:")
print(available_audit_columns)

display(histology_stage_inconsistency_records)

In [ ]:
# ============================================================
# Save the histology-stage inconsistency audit
# ============================================================

from pathlib import Path


PROJECT_DIRECTORY = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

TABLES_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Results"
    / "Tables"
)

TABLES_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


HISTOLOGY_STAGE_AUDIT_FILE = (
    TABLES_DIRECTORY
    / "histology_stage_inconsistency_audit.xlsx"
)


histology_stage_inconsistency_records.to_excel(
    HISTOLOGY_STAGE_AUDIT_FILE,
    index=False
)


# Verify that the file was created.
assert HISTOLOGY_STAGE_AUDIT_FILE.exists(), (
    "The audit file was not saved successfully."
)


print("HISTOLOGY-STAGE AUDIT SAVED")
print("----------------------------")
print(
    "Records:",
    len(histology_stage_inconsistency_records)
)
print(
    "File:",
    HISTOLOGY_STAGE_AUDIT_FILE
)

In [ ]:
# ============================================================
# Examine age at diagnosis by menopause status
# ============================================================

age_by_menopause_summary = (
    analysis_data
    .groupby(
        "Menopause_Status",
        observed=False
    )["Age_at_Diagnosis"]
    .agg(
        Patients="count",
        Minimum="min",
        First_Quartile=lambda x: x.quantile(0.25),
        Median="median",
        Mean="mean",
        Third_Quartile=lambda x: x.quantile(0.75),
        Maximum="max"
    )
    .reset_index()
)


numeric_summary_columns = [
    "Minimum",
    "First_Quartile",
    "Median",
    "Mean",
    "Third_Quartile",
    "Maximum"
]

age_by_menopause_summary[
    numeric_summary_columns
] = age_by_menopause_summary[
    numeric_summary_columns
].round(1)


print("AGE AT DIAGNOSIS BY MENOPAUSE STATUS")
print("------------------------------------")

display(age_by_menopause_summary)


# Display potentially unusual age-menopause combinations.
unusual_age_menopause = (
    (
        analysis_data["Menopause_Status"].eq("Pre")
        &
        analysis_data["Age_at_Diagnosis"].ge(60)
    )
    |
    (
        analysis_data["Menopause_Status"].eq("Post")
        &
        analysis_data["Age_at_Diagnosis"].le(35)
    )
)


unusual_age_menopause_records = (
    analysis_data.loc[
        unusual_age_menopause,
        [
            "Age_at_Diagnosis",
            "Menopause_Status",
            "Recurrence"
        ]
    ]
    .sort_values(
        by=[
            "Menopause_Status",
            "Age_at_Diagnosis"
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nPre-menopausal patients aged at least 60 or "
    "post-menopausal patients aged at most 35:",
    len(unusual_age_menopause_records)
)

display(unusual_age_menopause_records)

In [ ]:
# ============================================================
# Examine tumor size by summary stage
# ============================================================

tumor_size_by_stage_summary = (
    analysis_data
    .groupby(
        "Summary_Stage",
        observed=False
    )["Tumor_Size_mm"]
    .agg(
        Patients="size",
        Observed="count",
        Missing=lambda x: x.isna().sum(),
        Minimum="min",
        First_Quartile=lambda x: x.quantile(0.25),
        Median="median",
        Mean="mean",
        Third_Quartile=lambda x: x.quantile(0.75),
        Maximum="max"
    )
    .reset_index()
)


tumor_size_by_stage_summary[
    "Missing_Percentage"
] = (
    tumor_size_by_stage_summary["Missing"]
    / tumor_size_by_stage_summary["Patients"]
    * 100
).round(2)


numeric_summary_columns = [
    "Minimum",
    "First_Quartile",
    "Median",
    "Mean",
    "Third_Quartile",
    "Maximum"
]

tumor_size_by_stage_summary[
    numeric_summary_columns
] = tumor_size_by_stage_summary[
    numeric_summary_columns
].round(1)


column_order = [
    "Summary_Stage",
    "Patients",
    "Observed",
    "Missing",
    "Missing_Percentage",
    "Minimum",
    "First_Quartile",
    "Median",
    "Mean",
    "Third_Quartile",
    "Maximum"
]

tumor_size_by_stage_summary = (
    tumor_size_by_stage_summary[column_order]
)


print("TUMOR SIZE BY SUMMARY STAGE")
print("---------------------------")

display(tumor_size_by_stage_summary)


# ============================================================
# Kruskal-Wallis comparison across stage categories
# ============================================================

from scipy.stats import kruskal


stage_order = [
    "In_Situ",
    "Localized",
    "Regional",
    "Distant",
    "Unknown"
]


stage_tumor_size_groups = [
    analysis_data.loc[
        analysis_data["Summary_Stage"].eq(stage),
        "Tumor_Size_mm"
    ].dropna().to_numpy()
    for stage in stage_order
]


kruskal_statistic, kruskal_p_value = kruskal(
    *stage_tumor_size_groups
)


print("\nKRUSKAL-WALLIS TEST")
print("-------------------")
print(
    "Statistic:",
    round(float(kruskal_statistic), 3)
)
print(
    "Raw p value:",
    f"{kruskal_p_value:.6g}"
)


# ============================================================
# Display verified extreme tumor-size records
# ============================================================

extreme_tumor_size_records = (
    analysis_data.loc[
        analysis_data["Tumor_Size_mm"].ge(150),
        [
            "Tumor_Size_mm",
            "Primary_Site_Group",
            "Histology_Group",
            "Summary_Stage",
            "Recurrence"
        ]
    ]
    .sort_values(
        by="Tumor_Size_mm",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\nPatients with tumor size of at least 150 mm:",
    len(extreme_tumor_size_records)
)

display(extreme_tumor_size_records)

In [ ]:
# ============================================================
# Final continuous-predictor relationship checks
# ============================================================

from scipy.stats import kruskal, spearmanr


# ------------------------------------------------------------
# 1. Tumor size across known summary-stage categories
# ------------------------------------------------------------

known_stage_order = [
    "In_Situ",
    "Localized",
    "Regional",
    "Distant"
]


known_stage_tumor_size_groups = [
    analysis_data.loc[
        analysis_data["Summary_Stage"].eq(stage),
        "Tumor_Size_mm"
    ].dropna().to_numpy()
    for stage in known_stage_order
]


stage_statistic, stage_p_value = kruskal(
    *known_stage_tumor_size_groups
)


print("TUMOR SIZE ACROSS KNOWN SUMMARY-STAGE CATEGORIES")
print("------------------------------------------------")
print(
    "Kruskal-Wallis statistic:",
    round(float(stage_statistic), 3)
)
print(
    "Raw p value:",
    f"{stage_p_value:.6g}"
)


# ------------------------------------------------------------
# 2. Correlation between age and tumor size
# ------------------------------------------------------------

age_tumor_size_data = (
    analysis_data[
        [
            "Age_at_Diagnosis",
            "Tumor_Size_mm"
        ]
    ]
    .dropna()
)


overall_rho, overall_p_value = spearmanr(
    age_tumor_size_data["Age_at_Diagnosis"].to_numpy(),
    age_tumor_size_data["Tumor_Size_mm"].to_numpy()
)


correlation_results = [
    {
        "Group": "Overall",
        "Complete_Observations": len(age_tumor_size_data),
        "Spearman_Rho": overall_rho,
        "Raw_P_Value": overall_p_value
    }
]


# Calculate the correlation separately by recurrence group.
for recurrence_group in ["No", "Yes"]:

    group_data = (
        analysis_data.loc[
            analysis_data["Recurrence"].eq(recurrence_group),
            [
                "Age_at_Diagnosis",
                "Tumor_Size_mm"
            ]
        ]
        .dropna()
    )

    group_rho, group_p_value = spearmanr(
        group_data["Age_at_Diagnosis"].to_numpy(),
        group_data["Tumor_Size_mm"].to_numpy()
    )

    correlation_results.append(
        {
            "Group": recurrence_group,
            "Complete_Observations": len(group_data),
            "Spearman_Rho": group_rho,
            "Raw_P_Value": group_p_value
        }
    )


correlation_results = pd.DataFrame(
    correlation_results
)


correlation_results["Spearman_Rho"] = (
    correlation_results["Spearman_Rho"]
    .round(3)
)

correlation_results["Raw_P_Value"] = (
    correlation_results["Raw_P_Value"]
    .map(lambda value: f"{value:.6g}")
)


print("\nAGE AND TUMOR-SIZE CORRELATION")
print("------------------------------")

display(correlation_results)


# ------------------------------------------------------------
# 3. Verify continuous-variable ranges
# ------------------------------------------------------------

continuous_range_check = pd.DataFrame({
    "Variable": [
        "Age at diagnosis",
        "Tumor size"
    ],
    "Observed": [
        analysis_data["Age_at_Diagnosis"].notna().sum(),
        analysis_data["Tumor_Size_mm"].notna().sum()
    ],
    "Missing": [
        analysis_data["Age_at_Diagnosis"].isna().sum(),
        analysis_data["Tumor_Size_mm"].isna().sum()
    ],
    "Minimum": [
        analysis_data["Age_at_Diagnosis"].min(),
        analysis_data["Tumor_Size_mm"].min()
    ],
    "Maximum": [
        analysis_data["Age_at_Diagnosis"].max(),
        analysis_data["Tumor_Size_mm"].max()
    ]
})


print("\nFINAL CONTINUOUS-VARIABLE RANGE CHECK")
print("-------------------------------------")

display(continuous_range_check)

In [ ]:
# ============================================================
# Create the detailed supplementary participant-flow diagram
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch


# ------------------------------------------------------------
# Define output directory
# ------------------------------------------------------------

PROJECT_DIRECTORY = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

FIGURES_DIRECTORY = (
    PROJECT_DIRECTORY
    / "Results"
    / "Figures"
)

FIGURES_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Verified cohort and date-quality counts
# ------------------------------------------------------------

source_records = 2695

diagnosis_date_complete = 2386
diagnosis_date_approximate = 309
diagnosis_date_missing = 0

recurrence_date_complete = 107
recurrence_date_approximate = 6
recurrence_date_missing = 2582

eligible_cohort = 1370
excluded_after_cutoff = 1325

excluded_with_recurrence = 23

no_recurrence = 1297
recurrence = 73

candidate_predictors = 17


# ------------------------------------------------------------
# Validate all reported counts
# ------------------------------------------------------------

assert (
    diagnosis_date_complete
    + diagnosis_date_approximate
    + diagnosis_date_missing
    == source_records
), "Diagnosis-date counts do not reconcile."

assert (
    recurrence_date_complete
    + recurrence_date_approximate
    + recurrence_date_missing
    == source_records
), "Recurrence-date counts do not reconcile."

assert (
    eligible_cohort
    + excluded_after_cutoff
    == source_records
), "Eligibility counts do not reconcile."

assert (
    no_recurrence
    + recurrence
    == eligible_cohort
), "Outcome counts do not reconcile."


# ------------------------------------------------------------
# Set up the figure
# ------------------------------------------------------------

fig, axis = plt.subplots(
    figsize=(13, 16)
)

axis.set_xlim(0, 13)
axis.set_ylim(0, 18)
axis.axis("off")


# ------------------------------------------------------------
# Drawing functions
# ------------------------------------------------------------

def add_flow_box(
    x,
    y,
    width,
    height,
    text,
    facecolor="#EAF2F8",
    edgecolor="#1F4E79",
    fontsize=10.5,
    linestyle="-",
    linewidth=1.5
):
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle=(
            "round,pad=0.04,"
            "rounding_size=0.08"
        ),
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
        linestyle=linestyle
    )

    axis.add_patch(box)

    axis.text(
        x + width / 2,
        y + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color="#1F1F1F",
        linespacing=1.35,
        wrap=True
    )


def add_arrow(
    start,
    end,
    color="#1F4E79",
    linestyle="-"
):
    axis.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": 1.6,
            "color": color,
            "linestyle": linestyle,
            "shrinkA": 2,
            "shrinkB": 2
        }
    )


# ------------------------------------------------------------
# Figure title
# ------------------------------------------------------------

axis.text(
    6.5,
    17.55,
    "Participant Flow and Construction of the "
    "Five-Year Recurrence Cohort",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold",
    color="#1F1F1F"
)


# ------------------------------------------------------------
# Source registry
# ------------------------------------------------------------

add_flow_box(
    3.75,
    15.7,
    5.5,
    1.1,
    "Source breast cancer registry extract\n"
    f"(N = {source_records:,})",
    facecolor="#D6EAF8",
    edgecolor="#1F4E79",
    fontsize=11.5
)


# ------------------------------------------------------------
# Date standardization
# ------------------------------------------------------------

add_flow_box(
    3.75,
    13.9,
    5.5,
    1.0,
    "Diagnosis and recurrence dates standardized\n"
    "and assigned date-quality indicators",
    facecolor="#EBF5FB",
    edgecolor="#2874A6"
)

add_arrow(
    (6.5, 15.7),
    (6.5, 14.9)
)


# Diagnosis-date quality audit
add_flow_box(
    0.25,
    11.65,
    5.9,
    1.55,
    "Diagnosis-date quality\n"
    f"Complete: {diagnosis_date_complete:,}\n"
    f"Approximate: {diagnosis_date_approximate:,}\n"
    f"Missing or unparseable: {diagnosis_date_missing:,}",
    facecolor="#F8F9F9",
    edgecolor="#5D6D7E",
    fontsize=10
)


# Recurrence-date quality audit
add_flow_box(
    6.85,
    11.65,
    5.9,
    1.55,
    "Recurrence-date availability and quality\n"
    f"Complete: {recurrence_date_complete:,}\n"
    f"Approximate: {recurrence_date_approximate:,}\n"
    f"Missing: {recurrence_date_missing:,}",
    facecolor="#F8F9F9",
    edgecolor="#5D6D7E",
    fontsize=10
)


add_arrow(
    (5.4, 13.9),
    (3.2, 13.2),
    color="#5D6D7E",
    linestyle="--"
)

add_arrow(
    (7.6, 13.9),
    (9.8, 13.2),
    color="#5D6D7E",
    linestyle="--"
)


# Explanatory note
axis.text(
    6.5,
    11.2,
    "Date-quality categories were retained for auditing; "
    "they were not separate participant-exclusion groups.",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#566573",
    style="italic"
)


# ------------------------------------------------------------
# Five-year eligibility assessment
# ------------------------------------------------------------

add_flow_box(
    3.35,
    9.55,
    6.3,
    1.15,
    "Uniform five-year eligibility assessment\n"
    "Administrative study end: 1 September 2024\n"
    "Latest eligible diagnosis date: 1 September 2019",
    facecolor="#E8F8F5",
    edgecolor="#117864",
    fontsize=10.5
)

add_arrow(
    (6.5, 11.0),
    (6.5, 10.7)
)


# ------------------------------------------------------------
# Eligible and excluded branches
# ------------------------------------------------------------

add_flow_box(
    0.35,
    6.85,
    5.6,
    1.75,
    "Excluded from the primary cohort\n"
    "Diagnosis after 1 September 2019\n"
    f"(n = {excluded_after_cutoff:,})\n"
    "Five calendar years were unavailable "
    "before study closure",
    facecolor="#FDEDEC",
    edgecolor="#A93226",
    fontsize=10
)


add_flow_box(
    7.05,
    7.05,
    5.6,
    1.35,
    "Eligible five-year analysis cohort\n"
    "Diagnosis on or before 1 September 2019\n"
    f"(N = {eligible_cohort:,})",
    facecolor="#D5F5E3",
    edgecolor="#117864",
    fontsize=10.5
)


add_arrow(
    (5.0, 9.55),
    (3.2, 8.6),
    color="#A93226"
)

add_arrow(
    (8.0, 9.55),
    (9.85, 8.4),
    color="#117864"
)


# Excluded recurrence subgroup
add_flow_box(
    0.65,
    4.85,
    5.0,
    1.2,
    "Included within the excluded group:\n"
    "recurrence documented within five years\n"
    f"(n = {excluded_with_recurrence:,})",
    facecolor="#FADBD8",
    edgecolor="#C0392B",
    fontsize=9.8,
    linestyle="--"
)

add_arrow(
    (3.15, 6.85),
    (3.15, 6.05),
    color="#A93226",
    linestyle="--"
)


# ------------------------------------------------------------
# Final outcome branches
# ------------------------------------------------------------

add_flow_box(
    6.65,
    4.45,
    2.85,
    1.55,
    "No recurrence recorded\n"
    "within five years\n"
    f"(n = {no_recurrence:,}; 94.67%)",
    facecolor="#F4F6F7",
    edgecolor="#566573",
    fontsize=9.8
)


add_flow_box(
    9.8,
    4.45,
    2.85,
    1.55,
    "Recurrence recorded\n"
    "within five years\n"
    f"(n = {recurrence:,}; 5.33%)",
    facecolor="#FCF3CF",
    edgecolor="#B7950B",
    fontsize=9.8
)


add_arrow(
    (9.1, 7.05),
    (8.1, 6.0),
    color="#566573"
)

add_arrow(
    (10.6, 7.05),
    (11.2, 6.0),
    color="#B7950B"
)


# ------------------------------------------------------------
# Final modelling dataset
# ------------------------------------------------------------

add_flow_box(
    7.15,
    1.75,
    5.2,
    1.35,
    "Final modelling dataset\n"
    f"{eligible_cohort:,} patients\n"
    f"{candidate_predictors} candidate predictors "
    "and one binary outcome",
    facecolor="#E8DAEF",
    edgecolor="#6C3483",
    fontsize=10.5
)


add_arrow(
    (8.1, 4.45),
    (9.0, 3.1),
    color="#6C3483"
)

add_arrow(
    (11.2, 4.45),
    (10.5, 3.1),
    color="#6C3483"
)


# ------------------------------------------------------------
# Bottom note
# ------------------------------------------------------------

axis.text(
    6.5,
    0.65,
    "Approximate dates contained an imputed day and/or month "
    "and were retained with a date-quality flag. "
    "Patients diagnosed after the eligibility cutoff were "
    "excluded irrespective of recurrence status.",
    ha="center",
    va="center",
    fontsize=9,
    color="#566573",
    style="italic",
    wrap=True
)


plt.tight_layout()


# ------------------------------------------------------------
# Save PNG and PDF versions
# ------------------------------------------------------------

PARTICIPANT_FLOW_PNG = (
    FIGURES_DIRECTORY
    / "supplementary_participant_flow.png"
)

PARTICIPANT_FLOW_PDF = (
    FIGURES_DIRECTORY
    / "supplementary_participant_flow.pdf"
)


fig.savefig(
    PARTICIPANT_FLOW_PNG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    PARTICIPANT_FLOW_PDF,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

assert PARTICIPANT_FLOW_PNG.exists(), (
    "The PNG participant-flow figure was not saved."
)

assert PARTICIPANT_FLOW_PDF.exists(), (
    "The PDF participant-flow figure was not saved."
)


print("DETAILED PARTICIPANT-FLOW DIAGRAM SAVED")
print("---------------------------------------")
print("PNG:", PARTICIPANT_FLOW_PNG)
print("PDF:", PARTICIPANT_FLOW_PDF)